# Predicting 30-Day Hospital Readmission in Medicare Patients
## An Interpretable XGBoost Model on MIMIC-IV v3.1

**Authors:** Thiago Bandeira · Armando Gonzalez
**Institution:** Florida International University (Miami, FL)
**Mentor:** Dr. Christian Poellabauer
**Tracks:** Business Analytics (Thiago) · Artificial Intelligence (Armando)
**Date:** April 2026

**Contact:** tbati006@fiu.edu · agonz1689@fiu.edu

---

### Abstract

Thirty-day all-cause hospital readmission is a major quality-of-care metric for Medicare beneficiaries and is penalised financially through the CMS Hospital Readmissions Reduction Program. This project develops and evaluates a supervised machine-learning pipeline that estimates thirty-day readmission risk for **244,576 Medicare admissions** drawn from MIMIC-IV v3.1. A staged feature-engineering process across seven dataset versions (V1 → V7) produced a parsimonious set of **50 clinically-motivated features** spanning prior utilisation, comorbidity, medication complexity, clinical severity, and operational flow. Four gradient-boosting families (LightGBM, XGBoost, CatBoost, HistGradientBoosting) were each averaged across ten random seeds and an optional scipy-optimised blend of the four was also constructed. The blended ensemble reached **0.795 test AUROC**, but the final deployment candidate is the single XGBoost model at **0.793 AUROC**, selected on model-complexity grounds because it achieves effectively the same discrimination with a substantially simpler maintenance, explainability, and inference profile. The XGBoost model outperforms the LACE index by 0.109 AUROC and a published ClinicalBERT baseline by 0.079 AUROC, and SHAP explanations deliver both global and patient-level rationale for every prediction. The result is an interpretable risk-scoring tool that can be embedded in existing EHR workflows.

**Keywords:** 30-day readmission · Medicare · MIMIC-IV · gradient boosting · XGBoost · SHAP · healthcare analytics · interpretable ML.

---

### Notebook Structure

| Section | Content |
|---|---|
| 1 | Setup, imports, configuration |
| 2 | Motivation & Background |
| 3 | Problem Definition & Research Questions |
| 4 | Prior Art & Challenges |
| 5 | Data Source & Cohort Construction |
| 6 | Exploratory Data Analysis |
| 7 | Feature Engineering (V1 → V7) |
| 8 | Methods, Train/Test Protocol |
| 9 | Model Training: Baselines, GBMs, Deep Learning |
| 10 | 4-GBM Ensemble on V7 + Scipy-Optimized Blending |
| 11 | Results: ROC, Calibration, Benchmarks |
| 12 | SHAP Interpretability |
| 13 | Answering the Research Questions |
| 14 | Discussion, Limitations, Future Work |
| 15 | Contributions & Conclusions |
| 16 | References |

> **Note on version naming.** Throughout this notebook we use **V1 → V7** to match the feature-engineering progression reported in our final paper and presentation. **V7 is the final 50-feature parsimonious model** used for deployment. A larger 368-feature exploration, referred to as the **Feature Expansion Version**, was also evaluated as a ceiling check — it adds only +0.005 AUROC over V7 and is not deployed.


---
## 1. Setup

### 1.1 Package installation (run once)


In [ ]:
# Uncomment to install dependencies the first time
# !pip install pandas numpy matplotlib seaborn scipy scikit-learn pyarrow duckdb
# !pip install lightgbm xgboost catboost shap optuna
# !pip install torch  # optional, for the deep-learning exploration section

import sys
print(f"Python version: {sys.version.split()[0]}")


### 1.2 Imports & global style

In [ ]:
# ── CRITICAL: Windows OpenMP / LightGBM crash guard ─────────────────────────
# On Windows + conda, LightGBM ships its own LLVM OpenMP runtime while
# scikit-learn / numpy load libgomp or MKL's OpenMP. When both get loaded
# into the same process, the OS kills the kernel with exit code 3221225477
# (0xC0000005, access violation). These three env vars must be set BEFORE
# any model library (lightgbm / xgboost / catboost / sklearn) is imported.
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")           # prevent OMP race
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")   # allow co-existing OMPs
os.environ.setdefault("MKL_NUM_THREADS", "1")           # prevent MKL/OMP collision

import json
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats
from scipy.optimize import minimize

# Disable tqdm monitor thread — it interacts badly with LightGBM's OpenMP
# workers on Windows and is another common trigger of 0xC0000005 crashes.
try:
    import tqdm
    tqdm.tqdm.monitor_interval = 0
except Exception:
    pass

from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    roc_curve, precision_recall_curve, confusion_matrix,
    classification_report,
)
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", font_scale=1.05)

# -------------------------------------------------------------------------
# Global color palette — matches our Capstone presentation & report figures
# -------------------------------------------------------------------------
PALETTE = {
    "teal":     "#0F7B8A",  # primary — below-average / safe
    "gold":     "#F4B942",  # warning — above-average
    "coral":    "#E8636F",  # danger  — high-risk
    "mint":     "#00C9A7",  # accent  — new / parsimonious
    "blue":     "#4C72B0",  # class 0 / not readmitted
    "orange":   "#DD8452",  # class 1 / readmitted
    "purple":   "#8B5CF6",  # top-5 accent
    "gray":     "#888888",
}
BINARY_PALETTE = {0: PALETTE["blue"], 1: PALETTE["orange"]}
TOP5_COLORS = [PALETTE["teal"], PALETTE["mint"], PALETTE["gold"], PALETTE["coral"], PALETTE["purple"]]

print("Environment ready. Palette loaded.")


### 1.3 Project paths

In [ ]:
# --------------------------------------------------------------------------
# Update BASE_DIR below to match your local clone of the project.
# The default assumes the notebook lives in the project root alongside
# the Dataset/, Final Model Results/, and figure PNGs.
# --------------------------------------------------------------------------
BASE_DIR = Path(".").resolve()
DATA_DIR = BASE_DIR / "Dataset" / "mimic-parquet"
FIG_DIR  = BASE_DIR                # existing PNGs live in the project root
ART_DIR  = BASE_DIR / "Final Model Results"

# Progressive feature-engineering tables (produced upstream by our DuckDB pipeline)
# V10 = 184-column expansion set (superset for V7 feature selection)
# V7  = 50-feature parsimonious set — produced by Section 8.2 below from
#       V10 + raw admissions/omr tables (target encodings, PCA, interactions, logs).
PATHS = {
    "v1":  DATA_DIR / "training_table_v1.parquet",
    "v10": DATA_DIR / "training_table_v10.parquet",   # 184 cols — superset for V7
    "v7":  BASE_DIR / "training_table_v7.parquet",    # 50 cols — built in 8.2
    "admissions": DATA_DIR / "admissions.parquet",
    "omr":        DATA_DIR / "omr.parquet",
    "v7_importance": ART_DIR / "v7_feature_importance.csv",
}

TARGET = "readmit_30d"
ID_COLS   = ["subject_id", "hadm_id"]
DT_COLS   = ["admittime_dt", "dischtime_dt"]
SKIP_COLS = ID_COLS + DT_COLS + ["insurance", TARGET]
RANDOM_STATE = 42
TEST_SIZE    = 0.20

for name, path in PATHS.items():
    status = "found" if Path(path).exists() else "MISSING"
    print(f"  training_table_{name}: {path.name}  [{status}]")
print(f"\nTARGET: {TARGET}  |  random_state: {RANDOM_STATE}  |  test_size: {TEST_SIZE}")


---
## 2. Motivation & Background

Unplanned hospital readmission within 30 days of discharge is a recurring quality-of-care concern for Medicare beneficiaries, whose combination of advanced age, multimorbidity, and polypharmacy places them at substantially elevated risk.

- **30-day readmissions** are a major healthcare challenge, linked to high costs and poorer patient outcomes.
- **Medicare beneficiaries (age 65+)** are particularly vulnerable.
- Reducing readmissions is a national priority, reinforced by CMS through the **Hospital Readmissions Reduction Program (HRRP)** since 2013.
- **Data-driven prediction at discharge** can help primary-care providers identify at-risk patients early and enable timely interventions.

> **$26 B+** — the annual direct cost of unplanned Medicare readmissions *(Source: CMS HRRP).*

Traditional clinical scores such as **LACE** and **HOSPITAL** rely on a small number of static variables and, in external validation, rarely exceed an AUROC of 0.70 — leaving substantial predictive headroom that structured-data machine learning can close.


---
## 3. Problem Definition & Research Questions

### 3.1 Problem statement

| Aspect | Description |
|---|---|
| **Challenge** | Identifying Medicare patients at highest risk of 30-day readmission is difficult at the moment of discharge. |
| **Goal** | Develop a predictive model using the MIMIC-IV v3.1 dataset to assess readmission risk for Medicare beneficiaries. |
| **Outcome** | Deliver an interpretable risk-scoring tool that supports primary-care providers with targeted interventions. |
| **Impact** | Missed risk predictions → avoidable hospitalizations, higher costs, poorer patient outcomes. |

### 3.2 Research questions

- **RQ1.** Which features are most predictive of 30-day readmissions in Medicare beneficiaries?
- **RQ2.** Which modeling approach (statistical, boosting, deep learning, or ensemble) achieves the best predictive performance?
- **RQ3.** Can interpretable ML methods (e.g., SHAP) provide actionable insights for primary-care providers at discharge?


---
## 4. Prior Art & Challenges

### 4.1 Published baselines

| Study | Approach | AUROC |
|---|---|---|
| van Walraven et al., 2010 (CMAJ) | LACE clinical index | ≈ 0.684 |
| Huang et al., 2020 (arXiv) | ClinicalBERT + clinical notes | 0.714 |
| Gorishniy et al., 2021 (NeurIPS) | FT-Transformer (tabular deep learning) | reports competitive performance |
| Literature baselines | Single LightGBM / XGBoost on MIMIC | ≈ 0.76 |

### 4.2 Four challenges that shaped our design

1. **Severe class imbalance (≈ 79 / 21).** Disqualifies accuracy as a primary metric; motivates AUROC, average precision, and calibration.
2. **Patient overlap across admissions.** Risks information leakage — mitigated by grouped train/test splits on `subject_id`.
3. **Combinatorial feature-space growth.** Joining lab, medication, and operational tables required a disciplined selection pass to keep the deployed model interpretable.
4. **Missing social determinants of health + external-hospital readmissions.** Place a principled ceiling on achievable performance with the available data.


---
## 5. Data Source & Cohort Construction

### 5.1 Dataset — MIMIC-IV v3.1

- **Source:** MIMIC-IV v3.1 (PhysioNet) from Beth Israel Deaconess Medical Center, Boston, MA.
- **Full release:** 546,028 inpatient admissions across 364,627 unique patients (2008–2022) with >300 raw variables.
- **Key clinical modules used:** Admissions · Diagnoses (ICD-9/10) · Procedures · DRG Codes · Prescriptions · Lab Events · Vital Signs · ICU Stays · Transfers · Microbiology.

### 5.2 Cohort definition

- **Cohort:** 244,576 Medicare admissions (`insurance == "Medicare"`).
- **Target:** `readmit_30d` (binary) — 21.1% positive rate.
- **Label computation:** derived strictly from pre-discharge timestamps to prevent temporal leakage.

*Source: Johnson et al., "MIMIC-IV, a freely accessible electronic health record dataset," Scientific Data, 2023.*


In [ ]:
# Load V1 for exploratory analysis (V1 is the baseline 21-feature set)
df_v1 = pd.read_parquet(PATHS["v1"])

print(f"V1 dataset loaded: {df_v1.shape[0]:,} rows x {df_v1.shape[1]} columns")
print(f"Target prevalence: {df_v1[TARGET].mean():.1%} readmitted within 30 days")
print(f"Unique patients:  {df_v1['subject_id'].nunique():,}")

# Column groupings used throughout the EDA
CCI_FLAGS    = [c for c in df_v1.columns if c.startswith("cci_")]
BINARY_FLAGS = CCI_FLAGS + ["weekend_discharge", "polypharmacy_flag", "prior_any_ed_6m"]
CONTINUOUS   = ["age_at_admit", "los_days", "discharge_hour", "distinct_drugs",
                "prior_admissions_6m", "prior_mean_los_6m"]
CATEGORICAL  = ["admission_type", "admission_location", "discharge_location", "gender"]

print(f"\nFeature groups defined:")
print(f"  CCI flags:    {len(CCI_FLAGS)}  ({CCI_FLAGS})")
print(f"  Binary flags: {len(BINARY_FLAGS)}")
print(f"  Continuous:   {len(CONTINUOUS)}")
print(f"  Categorical:  {len(CATEGORICAL)}")


---
## 6. Exploratory Data Analysis

Our EDA covers 41 analyses across 17 sections. Here we highlight the findings that most directly informed feature engineering and modelling choices.

### 6.1 Class imbalance

The target is skewed ~79 / 21, which justifies AUROC (and average precision / calibration) rather than accuracy as headline metrics.

![Class imbalance](eda_class_imbalance.png)

> **Figure 1.** Class balance of the 30-day readmission target in the 244,576-admission Medicare cohort. A trivial always-negative classifier would reach 78.9% accuracy yet capture **zero** at-risk patients — the empirical basis for choosing AUROC as our headline metric.


In [ ]:
# ── 6.1 Class imbalance — reproducible figure ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

counts = df_v1[TARGET].value_counts().sort_index()
bars = axes[0].bar(
    ["Not Readmitted (0)", "Readmitted (1)"], counts.values,
    color=[BINARY_PALETTE[0], BINARY_PALETTE[1]], edgecolor="white", linewidth=1.5,
)
for bar, cnt in zip(bars, counts.values):
    axes[0].text(
        bar.get_x() + bar.get_width()/2, bar.get_height() + 1500,
        f"{cnt:,}\n({cnt/len(df_v1):.1%})",
        ha="center", fontsize=12, fontweight="bold",
    )
axes[0].set_title("Target Distribution: 30-Day Readmission", fontsize=14, fontweight="bold")
axes[0].set_ylabel("Count")
axes[0].set_ylim(0, max(counts.values) * 1.15)

axes[1].pie(
    counts.values, labels=["Not Readmitted", "Readmitted"],
    colors=[BINARY_PALETTE[0], BINARY_PALETTE[1]],
    autopct="%1.1f%%", startangle=90, textprops={"fontsize": 12},
)
axes[1].set_title("Class Proportion", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.savefig("eda_class_imbalance.png", dpi=150, bbox_inches="tight")
plt.show()

# Why accuracy fails — naïve baseline
prev = df_v1[TARGET].mean()
print("=" * 60)
print("NAIVE BASELINE: always predict 'Not Readmitted'")
print("=" * 60)
print(f"  Accuracy:  {1 - prev:.1%}   <-- looks great, but ...")
print(f"  Precision: 0.0%             <-- never identifies anyone at risk")
print(f"  Recall:    0.0%             <-- misses ALL readmissions")
print(f"  AUROC:     0.500            <-- equivalent to a coin flip")


### 6.2 Discharge-destination stratification

Discharge destination is one of the strongest univariate signals in the cohort and a natural target for care-coordination intervention.

![Readmission by discharge destination](discharge-readmission-chart.png)

> **Figure 2.** 30-day readmission rate by discharge destination (cohort average 21.1% shown as the dashed line). Rates span from 3.9% (Hospice) to 50.3% (Psychiatric Facility) — a 13× range that makes discharge location both highly predictive and highly actionable.

**Takeaways**

1. Discharge destination shows wide variation in readmission rates (3.9% → 50.3%), suggesting strong predictive potential.
2. Care-coordination interventions can target the 56,615 Home-Health patients readmitting at 25.1% — a large, operationally tractable subgroup.


In [ ]:
# ── 6.2 Readmission rate by discharge destination ──────────────────────────
disch = (df_v1.groupby("discharge_location")[TARGET]
         .agg(["mean", "count"])
         .rename(columns={"mean": "readmit_rate", "count": "n"})
         .sort_values("readmit_rate", ascending=True))
disch = disch[disch.n >= 50]   # suppress very small groups

bar_colors = [
    PALETTE["coral"] if r > 0.30
    else PALETTE["gold"] if r > 0.211
    else PALETTE["teal"]
    for r in disch.readmit_rate
]

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(range(len(disch)), disch.readmit_rate * 100,
               color=bar_colors, edgecolor="white")
ax.set_yticks(range(len(disch)))
ax.set_yticklabels(disch.index, fontsize=11)
ax.set_xlabel("Readmission Rate (%)", fontsize=13)
ax.set_title("30-Day Readmission Rate by Discharge Destination",
             fontsize=15, fontweight="bold")

# Reference line — cohort average
ax.axvline(x=21.1, color="gray", ls="--", lw=2, alpha=0.7)
ax.text(21.5, len(disch) - 0.5, "Cohort avg: 21.1%", fontsize=10, color="gray")

# Inline annotations
for i, (loc, row) in enumerate(disch.iterrows()):
    ax.text(row.readmit_rate * 100 + 0.5, i,
            f"{row.readmit_rate:.1%}  (n={row.n:,.0f})",
            va="center", fontsize=10, fontweight="bold")

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor=PALETTE["coral"], label="High risk (>30%)"),
    Patch(facecolor=PALETTE["gold"],  label="Above average (21.1–30%)"),
    Patch(facecolor=PALETTE["teal"],  label="Below average (<21.1%)"),
], loc="lower right", fontsize=10)

plt.tight_layout()
plt.savefig("eda_discharge_location.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nReadmission rate by discharge destination:")
print(disch.assign(pct=lambda d: (d.readmit_rate*100).round(1))[["pct", "n"]]
        .rename(columns={"pct": "readmit_%"}))


### 6.3 DRG code analysis

DRGs are CMS-defined categories that classify hospital stays by diagnosis, procedures, and severity. They determine reimbursement and — as SHAP later confirms — are one of the top-three predictors of readmission risk.

![DRG analysis](eda_drg_analysis.png)

> **Figure 3.** Top-20 DRG codes by volume (left) and top-15 DRG codes by readmission rate with N ≥ 500 (right). Colours highlight DRGs above / below the cohort 21.1% average.


In [ ]:
# ── 6.3 DRG code analysis ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Left: top DRGs by volume
top_drg = df_v1["drg_code"].value_counts().head(20)
colors  = [PALETTE["teal"] if i < 5 else "#B8C5D6" for i in range(len(top_drg))]
axes[0].barh(range(len(top_drg)), top_drg.values, color=colors)
axes[0].set_yticks(range(len(top_drg)))
axes[0].set_yticklabels([f"DRG {c}" for c in top_drg.index], fontsize=10)
axes[0].set_xlabel("Number of Admissions")
axes[0].set_title("Top-20 DRG Codes by Volume", fontsize=14, fontweight="bold")
axes[0].invert_yaxis()
for i, cnt in enumerate(top_drg.values):
    axes[0].text(cnt + 50, i, f"{cnt:,}", va="center", fontsize=9)

# Right: top DRGs by readmission rate (min N=500)
drg_stats = (df_v1.groupby("drg_code")[TARGET]
               .agg(["mean", "count"])
               .rename(columns={"mean": "readmit_rate", "count": "n"}))
drg_freq = (drg_stats[drg_stats.n >= 500]
              .sort_values("readmit_rate", ascending=False).head(15))

bar_colors = [
    PALETTE["coral"] if r > 0.30
    else PALETTE["gold"] if r > 0.211
    else PALETTE["teal"]
    for r in drg_freq.readmit_rate
]
axes[1].barh(range(len(drg_freq)), drg_freq.readmit_rate * 100, color=bar_colors)
axes[1].set_yticks(range(len(drg_freq)))
axes[1].set_yticklabels([f"DRG {c} (n={drg_freq.loc[c, 'n']:,.0f})"
                         for c in drg_freq.index], fontsize=10)
axes[1].set_xlabel("Readmission Rate (%)")
axes[1].set_title("Top-15 DRG Codes by Readmission Rate (N >= 500)",
                  fontsize=14, fontweight="bold")
axes[1].axvline(x=21.1, color="gray", ls="--", lw=2, alpha=0.7)
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig("eda_drg_analysis.png", dpi=150, bbox_inches="tight")
plt.show()


### 6.4 Continuous feature distributions by readmission status

![Continuous distributions](eda_continuous_distributions.png)

> **Figure 4.** Distributions of the six continuous features in V1 split by readmission label. LOS, prior-6m admissions, and distinct-drug counts show the clearest separation.


In [ ]:
# ── 6.4 Continuous feature distributions ───────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, col in zip(axes.flat, CONTINUOUS):
    for label, color in [(0, BINARY_PALETTE[0]), (1, BINARY_PALETTE[1])]:
        subset = df_v1[df_v1[TARGET] == label][col].dropna()
        ax.hist(subset, bins=50, alpha=0.6, color=color, density=True,
                label=f"Class {label}")
    ax.set_title(col, fontsize=12, fontweight="bold")
    ax.legend(fontsize=9)
plt.suptitle("Continuous Feature Distributions by Readmission Status",
             fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("eda_continuous_distributions.png", dpi=150, bbox_inches="tight")
plt.show()


### 6.5 Comorbidity prevalence and readmission uplift

![Comorbidity uplift](eda_comorbidity.png)

> **Figure 5.** Prevalence of the seven Charlson Comorbidity Index (CCI) flags in the Medicare cohort (left) and the uplift in 30-day readmission rate conditional on each comorbidity being present (right). Renal disease and CHF produce the largest absolute uplifts.


In [ ]:
# ── 6.5 Comorbidity prevalence & uplift ────────────────────────────────────
cci_rates = pd.DataFrame({
    "prevalence":   df_v1[CCI_FLAGS].mean() * 100,
    "readmit_if_1": [df_v1[df_v1[c] == 1][TARGET].mean() * 100 for c in CCI_FLAGS],
    "readmit_if_0": [df_v1[df_v1[c] == 0][TARGET].mean() * 100 for c in CCI_FLAGS],
})
cci_rates["uplift"] = cci_rates["readmit_if_1"] - cci_rates["readmit_if_0"]
cci_rates = cci_rates.sort_values("uplift", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].barh(cci_rates.index, cci_rates.prevalence, color=PALETTE["teal"])
axes[0].set_xlabel("Prevalence (%)")
axes[0].set_title("Comorbidity Prevalence", fontweight="bold")

axes[1].barh(cci_rates.index, cci_rates.uplift, color=PALETTE["coral"])
axes[1].set_xlabel("Readmission Rate Uplift (pp)")
axes[1].set_title("Readmission Uplift When Comorbidity Present", fontweight="bold")
axes[1].axvline(x=0, color="gray", ls="-", lw=0.5)
plt.tight_layout()
plt.savefig("eda_comorbidity.png", dpi=150, bbox_inches="tight")
plt.show()
print(cci_rates.round(1).to_string())


### 6.6 Point-biserial correlations

![Correlation](eda_correlation.png)

> **Figure 6.** Point-biserial correlation of each V1 feature with the 30-day readmission label. No single feature shows |r| > 0.20 — confirming that readmission is inherently multifactorial.


In [ ]:
# ── 6.6 Point-biserial correlation ─────────────────────────────────────────
from scipy.stats import pointbiserialr

num_cols = CONTINUOUS + BINARY_FLAGS
pb = {}
for col in num_cols:
    valid = df_v1[[col, TARGET]].dropna()
    r, _ = pointbiserialr(valid[TARGET], valid[col])
    pb[col] = r
pb = pd.Series(pb).sort_values()

fig, ax = plt.subplots(figsize=(10, 8))
colors = [PALETTE["coral"] if v > 0 else PALETTE["blue"] for v in pb.values]
ax.barh(pb.index, pb.values, color=colors)
ax.set_xlabel("Point-Biserial Correlation with Readmission", fontsize=12)
ax.set_title("Feature Correlation with 30-Day Readmission",
             fontsize=14, fontweight="bold")
ax.axvline(x=0, color="gray", ls="-", lw=0.5)
plt.tight_layout()
plt.savefig("eda_correlation.png", dpi=150, bbox_inches="tight")
plt.show()


---
## 7. Feature Engineering — Progressive Enrichment V1 → V7

We engineered features as a **staged process across seven dataset versions (V1 → V7)** so the marginal contribution of each clinical domain could be measured. V7 is the 50-feature parsimonious set we selected for deployment. A broader 368-feature exploration — the **Feature Expansion Version** — was also evaluated as a ceiling check; its +0.005 AUROC gain over V7 does not justify the operational overhead.

### Table 1 — Progressive feature-engineering summary

| Version | N features | New content added | Best AUROC |
|---|---|---|---|
| **V1** | 21 | Demographics, admission type/location, 7 CCI flags, LOS, meds, prior use, DRG | 0.707 |
| **V2** | 24 | Prior DRG / disposition, medication entropy (90d), LOS trend (180d) | 0.763 |
| **V3** | 28 | Missingness flags; recomputed LOS | 0.762 |
| **V4** | 33 | Age × CCI, LOS × CCI, age buckets, `cci_total` | 0.763 |
| **V5** | 36 | LOS × age, cci², log(LOS) | 0.763 |
| **V6** | 40 | Lab / med / diagnosis / procedure counts, ICU utilisation | 0.771 |
| **V7** | **50** | **5 target encodings + 5 clinical interactions (final / deployed)** | **0.793** |
| Feature Expansion Version (ceiling check) | 368 | Unpruned superset: pairwise interactions + extended encodings | 0.800 |

The two largest marginal AUROC gains arise at **V2** (+0.056, from temporal + medication signals) and **V7** (+0.022, from target encodings + clinical interactions). Beyond V7, a seven-fold feature expansion (to 368 columns in the Feature Expansion Version) buys only a further +0.005 AUROC — the empirical basis for adopting V7 as the final deployed modelling frontier.


### 7.1 Per-version description

**Version 1 — Baseline Feature Set (21 features)**
- Demographics: `age`, `gender`
- Admission info: admission type, location, discharge location
- 7 Charlson comorbidity flags: MI, CHF, diabetes, COPD, cerebrovascular, renal, cancer
- Stay context: length of stay, discharge hour, weekend discharge
- Medication burden: distinct drugs, polypharmacy flag
- Prior 6-month use: admissions, mean LOS, any ED
- Diagnosis-Related Group code

**Version 2 — Temporal & Complexity Signals (+3 → 24 features)**
- `last_drg_dispo` — prior DRG + discharge disposition combined category
- `med_entropy_90d` — Shannon entropy of medication classes in last 90 days
- `los_trend_180d` — mean change in LOS vs the prior admission

**Version 3 — Data Quality & LOS Refinement (+4 → 28 features)**
- `los_days` recomputed as `(discharge − admit) / 24`
- `drg_code_is_missing` — 1 if DRG code is null
- `discharge_location_is_missing`
- `los_days_is_missing`

**Version 4 — Clinical Severity & Interaction Features (+5 → 33 features)**
- `age_bucket` (`lt65`, `65–74`, `75–84`, `85+`)
- `cci_total` — sum of CCI flags
- `admission_type_ord` — ELECTIVE=0 → TRAUMA=3
- `age_cci_interaction`
- `los_cci_interaction`

**Version 5 — Nonlinear & Interaction Enhancements (+3 → 36 features)**
- `los_age_interaction` — `los_days × age_at_admit`
- `cci_total_sq` — squared comorbidity burden
- `log_los_days` — `log1p(los_days)`

**Version 6 — Clinical Feature Expansion (+4 → 40 features)**
- Diagnosis complexity: `n_diagnoses`
- Procedure complexity: `n_procedures`
- Medication burden: `n_meds_total`, `n_meds_unique`
- Lab intensity: `n_labs_total`, `n_lab_item_types`, `n_labs_abnormal`
- ICU utilisation: `icu_flag`, `icu_total_hrs`
- Patient history: `prior_admissions_all`

**Version 7 — Target Encodings + Clinical Interactions (+10 → 50 features; final)**
- 5 target encodings (out-of-fold, smoothed) for high-cardinality categoricals such as `drg_code`, `primary_dx_chapter`, `discharge_location`.
- 5 clinical interactions layered on top of V6 (risk-interaction terms identified during V2–V6 tuning).

Versions were engineered upstream using DuckDB queries on MIMIC-IV parquet files and persisted as `training_table_v{n}.parquet`. The notebook assumes these tables already exist.


### 7.2 Diminishing returns across versions

![Feature-count diminishing returns](feature_diminishing_returns.png)

> **Figure 7.** Test AUROC versus feature count across the V1 → V7 engineering progression. The V7 parsimonious set (50 features, mint) recovers 99.4% of the Feature Expansion Version's 368-feature ceiling (red) at a fraction of the operational complexity.


In [ ]:
# ── 7.2 Version progression + diminishing returns ──────────────────────────
versions = [
    ("V1", 21,  0.707, "Baseline"),
    ("V2", 24,  0.763, "+ Temporal / entropy"),
    ("V3", 28,  0.762, "+ Missingness flags"),
    ("V4", 33,  0.763, "+ Severity & interactions"),
    ("V5", 36,  0.763, "+ Non-linear transforms"),
    ("V6", 40,  0.771, "+ Clinical counts + ICU"),
    ("V7", 50,  0.795, "Parsimonious (deployed)"),
    ("Feature Expansion Version", 368, 0.800, "Ceiling check (not deployed)"),
]

fig, ax = plt.subplots(figsize=(13, 6.2))

pairs = sorted([(v[1], v[2]) for v in versions])
ax.plot([p[0] for p in pairs], [p[1] for p in pairs],
        color="gray", ls="--", lw=1.2, alpha=0.55, zorder=1)

for name, x, y, _ in versions:
    c = (PALETTE["mint"]  if name == "V7"                         else
         PALETTE["coral"] if name == "Feature Expansion Version"  else
         PALETTE["teal"])
    ax.scatter(x, y, s=180, color=c, zorder=5,
               edgecolors="white", linewidths=1.8)

# Minimal inline labels: version tag only
inline = {
    "V1": (10, -4, "left", "center"),
    "V2": (0, 10, "center", "bottom"),
    "V3": (0, -16, "center", "top"),
    "V4": (0, 10, "center", "bottom"),
    "V5": (0, -16, "center", "top"),
    "V6": (0, 10, "center", "bottom"),
    "V7": (-10, 6, "right", "bottom"),
    "Feature Expansion Version": (-10, -4, "right", "center"),
}
for name, x, y, _ in versions:
    dx, dy, ha, va = inline[name]
    label = ("V7 (50 feat)"                  if name == "V7"                        else
             "Feature Expansion\n(368 feat)" if name == "Feature Expansion Version" else
             name)
    ax.annotate(label, (x, y), xytext=(dx, dy),
                textcoords="offset points",
                fontsize=10.5, fontweight="bold", ha=ha, va=va)

# Callout arrow V7 -> Feature Expansion Version
ax.annotate("", xy=(340, 0.800), xytext=(55, 0.7955),
            arrowprops=dict(arrowstyle="-|>",
                            color=PALETTE["coral"], lw=1.6, alpha=0.9))
ax.text(135, 0.785,
        "Only +0.005 AUROC for ~7x more features",
        color=PALETTE["coral"], fontsize=11, fontweight="bold",
        ha="center")

# Side legend panel
legend_lines = [f"{v[0]:<4}  ({v[1]:>3} feat)   AUROC {v[2]:.3f}   {v[3]}"
                for v in versions]
ax.text(1.02, 0.98, "\n".join(legend_lines),
        transform=ax.transAxes, fontsize=9.5, fontfamily="monospace",
        va="top", ha="left",
        bbox=dict(boxstyle="round,pad=0.6", fc="#F7F7F7",
                  ec=PALETTE.get("gray", "#808080"), lw=0.8))

ax.set_xscale("log")
ax.set_xlim(18, 500)
ax.set_ylim(0.695, 0.810)
ax.set_xlabel("Number of features (log scale)", fontsize=12)
ax.set_ylabel("Test AUROC", fontsize=12)
ax.set_title("Diminishing Returns - AUROC vs. Feature Count (V1 -> V7)",
             fontsize=14, fontweight="bold")
ax.yaxis.set_major_formatter(mtick.FormatStrFormatter("%.3f"))
ax.grid(True, alpha=0.3, which="both")

plt.tight_layout()
plt.subplots_adjust(right=0.62)   # room for the side legend
plt.savefig("feature_diminishing_returns.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nKey result: V7 (50 features, AUROC=0.795) recovers "
      f"{0.795/0.800:.1%} of the Feature Expansion Version's performance "
      f"with only {50/368:.1%} of the features.")


---
## 8. Methods — Train / Test Protocol

### 8.1 Software stack

| Layer | Library |
|---|---|
| Data manipulation | `pandas`, `numpy`, `pyarrow`, `duckdb` |
| Preprocessing & CV | `scikit-learn` (`GroupShuffleSplit`, `GroupKFold`, target-encoders) |
| Gradient boosting | `lightgbm`, `xgboost`, `catboost`, `sklearn.ensemble.HistGradientBoostingClassifier` |
| Deep learning | `torch` (MLPs, GRU/LSTM hybrids, FT-Transformer) |
| Hyper-parameter tuning | `optuna` (100 trials per family) |
| Interpretability | `shap` |
| Visualisation | `matplotlib`, `seaborn` |

### 8.2 Train/test protocol

- **Patient-grouped 80 / 20 split** using `GroupShuffleSplit(groups=subject_id)` — no patient appears in both partitions.
- **Inner 80 / 20 train / validation split** for hyper-parameter tuning.
- **Target encoding** of high-cardinality categoricals with 5-fold out-of-fold folds (prevents leakage; retained natively for CatBoost).
- **Missing values** handled by the histogram-based tree algorithms directly, preserving the informative missingness signal.

### 8.3 Model-selection philosophy

Each of the four gradient-boosting families was trained across 10 random seeds and averaged to reduce variance. A scipy-optimised blend of the four was computed; the single XGBoost model scored 0.793 — within seed-level variance of the 0.795 blend — while requiring only one model artefact, one inference call, and one set of hyper-parameters. On the joint axes of discrimination and operational complexity, **the single XGBoost model is the deployment candidate**; the 4-GBM blend is retained as the discrimination ceiling.


### 8.2 Building the V7 50-feature parsimonious set

The V7 feature set is the top-50 features (by LightGBM importance) from the
184-column V10 expansion set, plus several features that were engineered *at
training time* rather than persisted to disk. This section builds the
full V7 feature table from the persisted V10 parquet + raw MIMIC tables.

**Engineering steps:**

1. **Base** — load `training_table_v10.parquet` (184 columns).
2. **Demographics join** — race and language from `admissions.parquet`.
3. **OMR features** — last pre-admission BMI and outpatient diastolic BP
   from `omr.parquet` via a memory-efficient `merge_asof`.
4. **Log transforms** — `log_prior_readmit_count`, `log_prior_admits_6m`,
   `log_time_since_discharge`.
5. **Squared** — `prior_admits_6m_sq`.
6. **Interactions** — 11 pairwise interactions between prior-admission counts,
   age, comorbidities, severity, LOS trend, and readmit-history flags.
7. **Derivable composite** — `los_per_prior_admit = los_days / (prior_admits_all + 1)`.
8. **Proxies for un-persisted order-time features** — `orders_first_24h`,
   `orders_last_6h`, `new_med_rate_48h`, `max_unit_los_days`, `discharge_surge`
   (documented approximations).
9. **Patient-grouped 80/20 split** — `GroupShuffleSplit(groups=subject_id)`.
10. **Comorbidity PCA** — fit on train-only, transform both — produces
    `comorbidity_pc1 … pc5` (top 5 components of Elixhauser flags).
11. **5-fold OOF target encoding** on train for `primary_dx_chapter`,
    `discharge_location`, `drg_code`, `last_drg_dispo`, `race` with Bayesian
    smoothing (α=20, prior=global positive rate ≈ 0.211). Test rows get the
    full-train refit to avoid leakage.
12. **Final selection** — retain the 50 columns listed in
    `Final Model Results/v7_feature_importance.csv`.

The resulting parquet is persisted on disk; subsequent notebook runs skip
feature-building and load it directly.


In [ ]:
# ── 8.2 V7 feature engineering — build from V10 + raw tables ───────────────
# Produces training_table_v7.parquet and v7_split.npz in the project root.
# Re-run this cell only if you want to regenerate the V7 features from scratch.

import gc
from sklearn.decomposition import PCA
from sklearn.model_selection import KFold

V7_PARQUET = PATHS["v7"]
V7_SPLIT   = BASE_DIR / "v7_split.npz"
V7_TOP50   = pd.read_csv(PATHS["v7_importance"])["feature"].dropna().tolist()

if V7_PARQUET.exists() and V7_SPLIT.exists():
    print(f"✓ Loading V7 feature table from disk.")
    df_v7    = pd.read_parquet(V7_PARQUET)
    _s       = np.load(V7_SPLIT, allow_pickle=True)
    train_idx, test_idx = _s["train_idx"], _s["test_idx"]
    feature_cols        = _s["feature_cols"].tolist()
else:
    print("Building V7 features (30–60s) ...")
    # 1. Base V10
    df = pd.read_parquet(PATHS["v10"])
    print(f"  [1/12] V10 base: {df.shape[0]:,} × {df.shape[1]}")

    # 2. Demographics
    adm = pd.read_parquet(PATHS["admissions"])[["hadm_id", "race", "language"]]
    df = df.merge(adm, on="hadm_id", how="left")
    print(f"  [2/12] joined race/language")

    # 3. OMR — bmi_last & bp_diastolic_outpatient via merge_asof
    omr = pd.read_parquet(PATHS["omr"], columns=["subject_id", "chartdate",
                                                   "result_name", "result_value"])
    omr["chartdate"] = pd.to_datetime(omr["chartdate"]).astype("datetime64[ns]")

    admit = df[["subject_id", "hadm_id", "admittime_dt"]].copy()
    admit["admittime_dt"] = pd.to_datetime(admit["admittime_dt"]).astype("datetime64[ns]")
    admit = admit.sort_values("admittime_dt")

    def _asof_last(src, value_col, new_name):
        s = src[["subject_id", "chartdate", value_col]].dropna(subset=[value_col]).sort_values("chartdate")
        m = pd.merge_asof(admit, s, by="subject_id",
                          left_on="admittime_dt", right_on="chartdate",
                          direction="backward")
        return m[["hadm_id", value_col]].rename(columns={value_col: new_name})

    bmi = omr[omr["result_name"].isin(["BMI (kg/m2)", "BMI"])].copy()
    bmi["bmi"] = pd.to_numeric(bmi["result_value"], errors="coerce")
    bmi = bmi[(bmi["bmi"] >= 12) & (bmi["bmi"] <= 80)]
    df = df.merge(_asof_last(bmi, "bmi", "bmi_last"), on="hadm_id", how="left")
    del bmi; gc.collect()

    bp = omr[omr["result_name"].str.startswith("Blood Pressure", na=False)].copy()
    sp = bp["result_value"].str.split("/", n=1, expand=True)
    bp["diastolic"] = pd.to_numeric(sp[1], errors="coerce") if sp.shape[1] >= 2 else np.nan
    bp = bp[(bp["diastolic"] >= 20) & (bp["diastolic"] <= 150)]
    df = df.merge(_asof_last(bp, "diastolic", "bp_diastolic_outpatient"), on="hadm_id", how="left")
    del bp, sp, omr, admit; gc.collect()
    print(f"  [3/12] OMR:  bmi cov {df['bmi_last'].notna().mean():.1%}  |  "
          f"bp cov {df['bp_diastolic_outpatient'].notna().mean():.1%}")

    # 4. Log transforms
    _f = lambda s, v=0: pd.to_numeric(s, errors='coerce').fillna(v)
    df["log_prior_readmit_count"] = np.log1p(_f(df["prior_readmission_count"]))
    df["log_prior_admits_6m"]     = np.log1p(_f(df["prior_admissions_6m"]))
    df["log_time_since_discharge"]= np.log1p(_f(df["time_since_last_discharge"], 9999))
    print(f"  [4/12] log transforms")

    # 5. Squared
    df["prior_admits_6m_sq"] = _f(df["prior_admissions_6m"]) ** 2
    print(f"  [5/12] squared terms")

    # 6. Interactions
    df["prior_admits_x_age"]       = _f(df["prior_admissions_6m"]) * _f(df["age_at_admit"])
    df["renal_risk_x_age"]         = _f(df["cci_renal"]) * _f(df["age_at_admit"])
    df["freq_x_recency"]           = _f(df["prior_admissions_all"]) / (_f(df["time_since_last_discharge"], 9999) + 1.0)
    df["high_risk_meds_x_age"]     = (_f(df.get("opioid_at_discharge", 0)) +
                                       _f(df.get("anticoag_at_discharge", 0))) * _f(df["age_at_admit"])
    df["los_trend_x_prior_6m"]     = _f(df["los_trend_180d"]) * _f(df["prior_admissions_6m"])
    df["los_trend_x_prior_admits"] = _f(df["los_trend_180d"]) * _f(df["prior_admissions_all"])
    df["severity_x_readmit"]       = _f(df.get("severity_composite", 0)) * _f(df.get("is_readmission_90d", 0))
    df["severity_x_lab_abnormal"]  = _f(df.get("severity_composite", 0)) * _f(df["lab_abnormal_rate"])
    df["anemia_x_prior_admits"]    = _f(df.get("elix_anemia_deficiency", 0)) * _f(df["prior_admissions_6m"])
    df["nonenglish_x_prior_admits"]= ((df["language"].fillna("UNK") != "ENGLISH").astype(int) *
                                       _f(df["prior_admissions_6m"]))
    df["rapid_tfr_x_readmit"]      = _f(df.get("n_service_changes", 0)) * _f(df.get("is_readmission_90d", 0))
    print(f"  [6/12] 11 interaction features")

    # 7. Derivable composite
    df["los_per_prior_admit"] = _f(df["los_days"]) / (_f(df["prior_admissions_all"]) + 1.0)
    print(f"  [7/12] los_per_prior_admit")

    # 8. Proxies for un-persisted time-granular features (documented)
    df["orders_first_24h"] = _f(df.get("orders_per_day", 0))                     # proxy ≈ daily order rate
    df["orders_last_6h"]   = _f(df.get("late_order_rate", 0)) * _f(df.get("n_late_orders", 0))
    df["new_med_rate_48h"] = _f(df.get("n_discharge_drugs", 0)) / _f(df["los_days"], 1).clip(lower=0.5)
    df["max_unit_los_days"]= _f(df.get("icu_total_hrs", 0)) / 24.0
    df["discharge_surge"]  = ((_f(df.get("discharge_hour", 12)) >= 10) &
                              (_f(df.get("discharge_hour", 12)) <= 14)).astype(int)
    print(f"  [8/12] 5 proxy features")

    # 9. Patient-grouped split
    y_all  = df[TARGET].astype(int).values
    groups = df["subject_id"].values
    splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    train_idx, test_idx = next(splitter.split(df, y_all, groups=groups))
    assert not (set(groups[train_idx]) & set(groups[test_idx])), "patient leakage!"
    print(f"  [9/12] split: train {len(train_idx):,} | test {len(test_idx):,}")

    # 10. Comorbidity PCA (fit on train only)
    elix_cols = [c for c in df.columns if c.startswith("elix_")]
    elix_mat  = df[elix_cols].fillna(0).astype(float).values
    pca = PCA(n_components=5, random_state=RANDOM_STATE)
    pca.fit(elix_mat[train_idx])
    pcs = pca.transform(elix_mat)
    for i in range(5):
        df[f"comorbidity_pc{i+1}"] = pcs[:, i]
    print(f"  [10/12] comorbidity PCA  explained var = {pca.explained_variance_ratio_[:5].round(3).tolist()}")

    # 11. 5-fold OOF target encoding
    TE_COLS = ["primary_dx_chapter", "discharge_location", "drg_code", "last_drg_dispo", "race"]
    GLOBAL_PRIOR = float(y_all[train_idx].mean())
    SMOOTH = 20.0

    def _te(col):
        x_all = df[col].astype(str).fillna("__NA__").values
        out = np.full(len(df), GLOBAL_PRIOR, dtype=np.float32)
        x_tr, y_tr = x_all[train_idx], y_all[train_idx]
        kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
        for ftr, fva in kf.split(x_tr):
            agg = (pd.DataFrame({"x": x_tr[ftr], "y": y_tr[ftr]})
                   .groupby("x")["y"].agg(["sum", "count"]).reset_index())
            agg["enc"] = (agg["sum"] + SMOOTH * GLOBAL_PRIOR) / (agg["count"] + SMOOTH)
            mp = dict(zip(agg["x"], agg["enc"]))
            out[train_idx[fva]] = [mp.get(v, GLOBAL_PRIOR) for v in x_tr[fva]]
        # refit on full train → apply to test
        agg = (pd.DataFrame({"x": x_tr, "y": y_tr})
               .groupby("x")["y"].agg(["sum", "count"]).reset_index())
        agg["enc"] = (agg["sum"] + SMOOTH * GLOBAL_PRIOR) / (agg["count"] + SMOOTH)
        mp = dict(zip(agg["x"], agg["enc"]))
        out[test_idx] = [mp.get(v, GLOBAL_PRIOR) for v in x_all[test_idx]]
        return out

    for c in TE_COLS:
        df[f"{c}_te"] = _te(c)
    print(f"  [11/12] target encodings: {', '.join(c + '_te' for c in TE_COLS)}")

    # 12. Select V7 top-50 and persist to disk
    feature_cols = [f for f in V7_TOP50 if f in df.columns]
    missing = [f for f in V7_TOP50 if f not in df.columns]
    keep = ["subject_id", "hadm_id", "admittime_dt", "dischtime_dt", "insurance", TARGET] + feature_cols
    df_v7 = df[[c for c in keep if c in df.columns]].copy()
    df_v7.to_parquet(V7_PARQUET, index=False)
    np.savez(V7_SPLIT, train_idx=train_idx, test_idx=test_idx,
             feature_cols=np.array(feature_cols, dtype=object))
    print(f"  [12/12] selected {len(feature_cols)}/50 V7 features  "
          f"(missing: {missing if missing else 'none'})")
    print(f"  → saved {V7_PARQUET.name} and v7_split.npz")

# --- Build X_train, X_test, y_train, y_test, groups_train -------------------
X = df_v7[feature_cols].copy()
y = df_v7[TARGET].astype(int).values
groups = df_v7["subject_id"].values

# Label-encode remaining string categoricals (drg_code, discharge_location etc.)
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le

X_train, X_test = X.iloc[train_idx].reset_index(drop=True), X.iloc[test_idx].reset_index(drop=True)
y_train, y_test = y[train_idx], y[test_idx]
groups_train = groups[train_idx]

assert not (set(groups_train) & set(groups[test_idx])), "PATIENT LEAKAGE DETECTED"
print(f"\nTrain: {len(X_train):,} admissions  |  pos rate = {y_train.mean():.4f}")
print(f"Test:  {len(X_test):,} admissions  |  pos rate = {y_test.mean():.4f}")
print(f"Features: {len(feature_cols)}  ({len(cat_cols)} categorical)")


---
## 9. Model Training — Baselines → Deep Learning

Before the final 4-GBM ensemble, we ran the per-family progression V1 → V6 to isolate the marginal benefit of feature enrichment independently of model choice.

### 9.1 Logistic Regression V1 → V6 — baseline

Regularised logistic regression **peaks at V1 (0.6994) and deteriorates from V2 onward**. A linear model cannot exploit the richer non-linear signals added in V2–V6 — confirming that non-linear learners are required.

| Dataset | LogReg AUROC |
|---|---|
| V1 | **0.6994** |
| V2 | 0.6890 |
| V3 | 0.6885 |
| V4 | 0.6879 |
| V5 | 0.6870 |
| V6 | 0.6862 |


In [ ]:
# ── 9.1 Logistic regression V1 → V6 progression ────────────────────────────
# Reference values captured from our training runs (reproduce by iterating
# through PATHS for each version with a fresh LogisticRegression).
logreg_progression = {
    "V1": 0.6994, "V2": 0.6890, "V3": 0.6885,
    "V4": 0.6879, "V5": 0.6870, "V6": 0.6862,
}

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(list(logreg_progression.keys()), list(logreg_progression.values()),
        "o-", color=PALETTE["blue"], lw=2.5, markersize=10)
ax.axhline(0.70, color="gray", ls="--", lw=1)
for v, s in logreg_progression.items():
    ax.text(v, s + 0.0015, f"{s:.4f}", ha="center", fontsize=10, fontweight="bold")
ax.set_ylim(0.680, 0.710)
ax.set_ylabel("Test AUROC")
ax.set_title("Logistic Regression — test AUROC across V1–V6",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("fig_b_logreg_v1v6.png", dpi=150, bbox_inches="tight")
plt.show()


### 9.2 LightGBM V1 → V6

LightGBM **jumps from 0.709 at V1 to 0.759 at V2** as temporal and medication-complexity signals are introduced, holds steady through V3–V5, and rises to 0.769 at V6 with aggregate clinical counts + ICU utilisation.

### 9.3 XGBoost V1 → V6

Near-identical trajectory to LightGBM (V1 0.707 → V6 0.771) — the feature-engineering gains generalise across boosting implementations and are not an artefact of one library.

### 9.4 MLP V1 → V6

A two-layer [128, 64] MLP remains flat at ≈ 0.700 across the entire progression — default MLPs are **not competitive with boosting** on this tabular problem without dedicated tabular-deep-learning architectures.


In [ ]:
# ── 9.2/9.3/9.4 Per-family progression on V1 → V6 ──────────────────────────
progression = {
    "LightGBM": {"V1": 0.7090, "V2": 0.7590, "V3": 0.7610, "V4": 0.7630, "V5": 0.7650, "V6": 0.7690},
    "XGBoost":  {"V1": 0.7070, "V2": 0.7629, "V3": 0.7620, "V4": 0.7640, "V5": 0.7660, "V6": 0.7711},
    "MLP":      {"V1": 0.6980, "V2": 0.6990, "V3": 0.7000, "V4": 0.6995, "V5": 0.7010, "V6": 0.7020},
}

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)
for ax, (name, series) in zip(axes, progression.items()):
    color = {"LightGBM": PALETTE["teal"], "XGBoost": PALETTE["mint"], "MLP": PALETTE["gold"]}[name]
    ax.plot(list(series.keys()), list(series.values()), "o-", color=color, lw=2.5, markersize=10)
    for v, s in series.items():
        ax.text(v, s + 0.002, f"{s:.3f}", ha="center", fontsize=9, fontweight="bold")
    ax.set_title(f"{name} — V1 → V6", fontsize=13, fontweight="bold")
    ax.set_ylabel("Test AUROC")
    ax.set_ylim(0.68, 0.80)
plt.suptitle("Per-family AUROC progression on the V1 → V6 engineering staircase",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("fig_cde_lgbm_xgb_mlp_v1v6.png", dpi=150, bbox_inches="tight")
plt.show()


### 9.5 Advanced Neural Networks — FT-Transformer, GRU+MLP, Stacking (V6)

We also evaluated attention-based and sequence-based architectures on the V6 dataset.

#### FT-Transformer (Gorishniy et al., 2021)

*How it works:*
1. **Feature tokenizer** — numeric + categorical features → individual *d*-dimensional vector
2. **[CLS] token** — learnable summary token prepended to the feature tokens
3. **Self-attention** — every token attends to all others, discovering which features interact
4. **Classification head** — [CLS] output → LayerNorm → Dense(64) → Sigmoid → readmission probability

*Why it adds value:* unlike MLPs (which blend features implicitly through layers) or LightGBM (which finds interactions locally via greedy tree splits), FT-Transformer **learns explicitly which feature pairs matter via attention** — global and differentiable.

#### GRU + MLP Hybrid

| Branch | Inputs |
|---|---|
| **Sequence branch** | 10 prior admissions × 15 temporal features → GRU layers → temporal representation |
| **Static branch**   | 25 numeric features + 5 categorical embeddings → Dense layers |
| **Fusion**          | Concatenate → Dense(128) → Dense(64) → Sigmoid |

**Key finding:** standalone RNNs (AUROC 0.636) confirm that sequences alone are insufficient — the hybrid approach combining temporal and static features is essential.

#### Test AUROC comparison on V6

| Model family | Best model | AUROC |
|---|---|---|
| Hybrid GRU | `HybridGRU_1L32` | 0.7711 |
| Hybrid LSTM | `Hybrid_LSTM32` | 0.7695 |
| Standalone RNN | `BiGRU_2L64` | 0.6365 |
| FT-Transformer | Medium | ≈ 0.769 |
| Embedding MLP | Deep | 0.762 |
| **Stacking (LR meta)** | 4×GRU + 2×MLP + 3×FT-Trans + 2×LGB | **0.7778** |


In [ ]:
# ── 9.5 V6 cross-family comparison ─────────────────────────────────────────
v6_comparison = {
    "MLP baseline":    0.7020,
    "FT-Transformer":  0.7690,
    "Hybrid LSTM":     0.7695,
    "Hybrid GRU+MLP":  0.7711,
    "LightGBM":        0.7690,
    "XGBoost":         0.7711,
    "Standalone BiGRU": 0.6365,
    "Stacking (LR meta)": 0.7778,
}
v6_sorted = dict(sorted(v6_comparison.items(), key=lambda x: x[1]))

fig, ax = plt.subplots(figsize=(10, 5))
colors = [PALETTE["coral"] if v < 0.70
          else PALETTE["gold"] if v < 0.77
          else PALETTE["teal"] if v < 0.778
          else PALETTE["mint"]
          for v in v6_sorted.values()]
ax.barh(list(v6_sorted.keys()), list(v6_sorted.values()), color=colors)
ax.axvline(0.778, color="gray", ls="--", lw=1,
           label="Stacking ceiling 0.778")
for i, v in enumerate(v6_sorted.values()):
    ax.text(v + 0.002, i, f"{v:.4f}", va="center", fontsize=10, fontweight="bold")
ax.set_xlim(0.62, 0.80)
ax.set_xlabel("Test AUROC")
ax.set_title("V6 cross-family comparison (244,576 admissions, patient-grouped)",
             fontsize=13, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig("fig_f_v6_families.png", dpi=150, bbox_inches="tight")
plt.show()


### 9.6 Why Gradient Boosting for V7?

Four reasons we selected GBMs as the V7 architecture:

1. **V7 features are all static tabular.** GRU/LSTM hybrids exploited 10 prior admissions × 15 temporal features; V7's new features (target encodings + clinical interactions) are static, so there are no sequences to leverage.
2. **Diminishing returns from neural nets.** The best V6 neural ensemble (0.778) beat the best single GBM (0.769) by only +0.009 — neural nets did not outperform boosting on V6 at a level that justifies their operational cost.
3. **Interpretability required (RQ3).** GBMs support full SHAP attribution, which is essential for actionable discharge insights.
4. **V7 = 50 tabular features** (V6's 40 + 5 target encodings + 5 clinical interactions). Ideal for GBM architectures.

**Conclusion:** GBMs are the optimal architecture for V7 — matching the data structure, preserving interpretability, and achieving 0.795 AUROC in the blend (and 0.793 for the single XGBoost).


---
## 10. Four-GBM Ensemble on V7 + Scipy-Optimized Blending

### 10.1 Per-model architecture summary

| Model | Key strength | V7 AUROC (10-seed avg, reported) |
|---|---|---|
| **LightGBM**     | Histogram-based, leaf-wise growth, fast training, handles categoricals | 0.790 |
| **XGBoost**      | Lossguide tree policy, best individual performer                        | **0.793** |
| **CatBoost**     | Ordered boosting, native categorical handling                           | 0.792 |
| **HistGBM**      | Sklearn-native histogram gradient, native missing-value support         | 0.792 |

The report numbers above came from a 10-seed training run with
**Optuna-tuned hyperparameters** (150 trials per family) on an RTX 4080 GPU.
This notebook re-creates that pipeline end-to-end.

The `v7_best_params.json` artefact produced by Section 10.2 is the reproduction
anchor — every subsequent run loads those exact params and should reproduce
the ensemble AUROCs to within ±0.001.


### 10.2 Optuna hyperparameter tuning (150 trials × 3 families)

We re-tune LightGBM, XGBoost, and CatBoost on the V7 50-feature set using
TPE with a patient-grouped validation split. HistGBM uses sklearn defaults
(it has fewer knobs and is only the fourth blend component).

**Run time (default `N_TRIALS = 50`):**
- **GPU (RTX 4080):** ~15–25 min total for all three families
- **CPU (Windows, n_jobs=1):** ~2–4 hours total

Set `N_TRIALS = 150` to exactly match the report's grid density (adds ~3× time).

**Caching:** results are persisted to `v7_best_params.json`. Re-running this
cell with the file present **skips tuning** and loads cached params, so the
ensemble below is always reproducible without re-tuning.

**Windows stability:** the setup cell forces `OMP_NUM_THREADS=1`,
`MKL_NUM_THREADS=1`, `KMP_DUPLICATE_LIB_OK=TRUE` and all model calls run with
`n_jobs=1` — this is what the original V10 pipeline used to prevent the
LightGBM OpenMP access-violation crash (exit code `3221225477` / `0xC0000005`).
Do not raise `n_jobs` on Windows unless you've verified your env doesn't load
both libgomp and LLVM OpenMP.


In [ ]:
# ── 10.2 Optuna tuning → v7_best_params.json ────────────────────────────────
# Tunes LightGBM, XGBoost, CatBoost on the 50-feature V7 set with 150 trials
# each (TPE). Saves best_params to disk; subsequent runs load from cache.
#
# If v7_best_params.json already exists next to this notebook, Optuna and the
# retraining libraries are NOT imported — the cached params are loaded and the
# cell exits immediately. This lets the notebook run without optuna installed.

import json, time, os

# --- Toggles ---------------------------------------------------------------
N_TRIALS      = 50        # report used 150; 50 gives ~95% of the benefit and
                          # is much safer on a Windows/LightGBM kernel.
                          # Set to 150 to exactly reproduce the report's grid.
USE_GPU       = False     # set True if you have CUDA + xgboost/catboost GPU builds
FORCE_RETUNE  = False     # set True to ignore cached params and re-run Optuna
PARAMS_PATH   = Path("v7_best_params.json")

# --- Patient-grouped validation split (shared by Optuna and the ensemble) ---
splitter_val = GroupShuffleSplit(n_splits=1, test_size=0.10, random_state=0)
tr_idx, va_idx = next(splitter_val.split(X_train, y_train, groups=groups_train))
X_tr, X_va = X_train.iloc[tr_idx].reset_index(drop=True), X_train.iloc[va_idx].reset_index(drop=True)
y_tr, y_va = y_train[tr_idx], y_train[va_idx]
print(f"Train→fit: {len(X_tr):,}  |  Train→val: {len(X_va):,}  |  Test: {len(X_test):,}")

# --- Class-imbalance scale for LightGBM/XGBoost (≈ 3.74 for 21% positive) ---
spw = float((y_tr == 0).sum() / max((y_tr == 1).sum(), 1))
print(f"scale_pos_weight (neg/pos ratio on train): {spw:.3f}")

# --- Skip tuning if cached (no optuna import needed!) -----------------------
if PARAMS_PATH.exists() and not FORCE_RETUNE:
    best_params = json.loads(PARAMS_PATH.read_text())
    # strip provenance metadata before the training cells consume these
    best_params = {k: v for k, v in best_params.items() if not k.startswith("_")}
    print(f"\n✓ Loaded cached best_params from {PARAMS_PATH} — skipping tuning.")
    print(f"  families: {list(best_params.keys())}")
else:
    # Retuning path — imports happen here so the cached-only path doesn't need them
    print("\nNo cached params (or FORCE_RETUNE=True) → running Optuna.")
    print("Required packages: optuna, lightgbm, xgboost, catboost")
    print("Install with:  %pip install optuna lightgbm xgboost catboost --quiet")
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    import lightgbm as lgb
    from xgboost import XGBClassifier
    from catboost import CatBoostClassifier

    def _progress_cb(every=5):
        def cb(study, trial):
            if (trial.number + 1) % every == 0 or trial.number == 0:
                print(f"    trial {trial.number+1:>3}/{N_TRIALS}  "
                      f"best val AUROC = {study.best_value:.4f}", flush=True)
        return cb

    best_params = {}

    # --- LightGBM Optuna ----------------------------------------------------
    print("\n" + "=" * 64)
    print(f"LightGBM Optuna ({N_TRIALS} trials)")
    print("=" * 64)
    t0 = time.time()
    def lgb_obj(trial):
        p = {
            "objective": "binary", "metric": "auc", "verbosity": -1,
            "random_state": 42, "n_jobs": 1,
            "num_leaves":        trial.suggest_int("num_leaves", 20, 300),
            "max_depth":         trial.suggest_int("max_depth", 3, 14),
            "min_child_samples": trial.suggest_int("min_child_samples", 20, 300),
            "learning_rate":     trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
            "n_estimators":      trial.suggest_int("n_estimators", 500, 3000),
            "subsample":         trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.3, 1.0),
            "reg_alpha":         trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda":        trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
            "scale_pos_weight":  trial.suggest_float("scale_pos_weight", 1.0, 6.0),
        }
        if USE_GPU:
            p["device"] = "gpu"
        m = lgb.LGBMClassifier(**p)
        m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
              callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])
        return roc_auc_score(y_va, m.predict_proba(X_va)[:, 1])
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(lgb_obj, n_trials=N_TRIALS, show_progress_bar=False,
                   callbacks=[_progress_cb(5)])
    best_params["lgb"] = study.best_params
    print(f"  best val AUROC = {study.best_value:.4f}  ({time.time()-t0:.0f}s)")

    # --- XGBoost Optuna -----------------------------------------------------
    print("\n" + "=" * 64)
    print(f"XGBoost Optuna ({N_TRIALS} trials)")
    print("=" * 64)
    t0 = time.time()
    def xgb_obj(trial):
        p = {
            "objective": "binary:logistic", "eval_metric": "auc",
            "verbosity": 0, "random_state": 42, "n_jobs": 1,
            "tree_method": "hist", "grow_policy": "lossguide",
            "early_stopping_rounds": 100,
            "max_depth":         trial.suggest_int("max_depth", 3, 14),
            "learning_rate":     trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
            "n_estimators":      trial.suggest_int("n_estimators", 500, 3000),
            "subsample":         trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.3, 1.0),
            "min_child_weight":  trial.suggest_int("min_child_weight", 1, 100),
            "gamma":             trial.suggest_float("gamma", 0.0, 5.0),
            "reg_alpha":         trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda":        trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
            "scale_pos_weight":  trial.suggest_float("scale_pos_weight", 1.0, 6.0),
        }
        if USE_GPU:
            p["device"] = "cuda"
        m = XGBClassifier(**p)
        m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        return roc_auc_score(y_va, m.predict_proba(X_va)[:, 1])
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(xgb_obj, n_trials=N_TRIALS, show_progress_bar=False,
                   callbacks=[_progress_cb(5)])
    best_params["xgb"] = study.best_params
    print(f"  best val AUROC = {study.best_value:.4f}  ({time.time()-t0:.0f}s)")

    # --- CatBoost Optuna ----------------------------------------------------
    print("\n" + "=" * 64)
    print(f"CatBoost Optuna ({N_TRIALS} trials)")
    print("=" * 64)
    t0 = time.time()
    def cb_obj(trial):
        p = {
            "loss_function": "Logloss", "eval_metric": "AUC",
            "random_seed": 42, "verbose": 0,
            "iterations":       trial.suggest_int("iterations", 500, 3000),
            "depth":            trial.suggest_int("depth", 4, 10),
            "learning_rate":    trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
            "l2_leaf_reg":      trial.suggest_float("l2_leaf_reg", 1.0, 20.0),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 2.0),
            "random_strength":  trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
            "border_count":     trial.suggest_int("border_count", 64, 254),
        }
        if USE_GPU:
            p["task_type"] = "GPU"; p["devices"] = "0"
        m = CatBoostClassifier(**p, early_stopping_rounds=100)
        m.fit(X_tr, y_tr, eval_set=(X_va, y_va), verbose=0)
        return roc_auc_score(y_va, m.predict_proba(X_va)[:, 1])
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(cb_obj, n_trials=N_TRIALS, show_progress_bar=False,
                   callbacks=[_progress_cb(5)])
    best_params["cb"] = study.best_params
    print(f"  best val AUROC = {study.best_value:.4f}  ({time.time()-t0:.0f}s)")

    # --- Persist ------------------------------------------------------------
    PARAMS_PATH.write_text(json.dumps(best_params, indent=2))
    print(f"\n✓ Saved best_params → {PARAMS_PATH.resolve()}")

# Display loaded/tuned params (first few keys)
for family, p in best_params.items():
    print(f"\n{family.upper()}:", {k: (round(v, 4) if isinstance(v, float) else v)
                                    for k, v in list(p.items())[:6]}, "...")


### 10.3 4-GBM multi-seed ensemble — training with tuned params

Trains each family with the `v7_best_params.json` hyperparameters across
`N_SEEDS` random seeds and averages the predictions. Memory is reclaimed
with `gc.collect()` between seeds to prevent OOM on the 244K-row data.

**Reproduction note:** the report used `N_SEEDS = 10`. This notebook defaults
to `N_SEEDS = 3` for faster re-runs — a 3-seed average typically lands within
±0.002 AUROC of the 10-seed value.


In [ ]:
# ── 10.3 4-GBM multi-seed ensemble on V7 (uses v7_best_params.json) ─────────
import gc
import lightgbm as lgb
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

N_SEEDS = 3   # set to 10 to exactly match the report (and ~3x runtime)

# --- LightGBM ---------------------------------------------------------------
lgb_base = {
    **best_params["lgb"],
    "objective": "binary", "metric": "auc",
    "verbosity": -1, "n_jobs": 1,
}
if USE_GPU: lgb_base["device"] = "gpu"

print(f"\nTraining LightGBM across {N_SEEDS} seeds ...")
lgb_preds = np.zeros((len(X_test), N_SEEDS))
for s in range(N_SEEDS):
    p = {**lgb_base, "random_state": s * 42}
    m = lgb.LGBMClassifier(**p)
    m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
          callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])
    lgb_preds[:, s] = m.predict_proba(X_test)[:, 1]
    if s == 0:
        lgb_model_seed0 = m   # keep for SHAP later
    else:
        del m
    gc.collect()
lgb_avg = lgb_preds.mean(axis=1)
lgb_auroc = roc_auc_score(y_test, lgb_avg)
print(f"  LightGBM {N_SEEDS}-seed avg AUROC = {lgb_auroc:.4f}")


In [ ]:
# --- XGBoost ----------------------------------------------------------------
xgb_base = {
    **best_params["xgb"],
    "objective": "binary:logistic", "eval_metric": "auc",
    "tree_method": "hist", "grow_policy": "lossguide",
    "verbosity": 0, "n_jobs": 1, "early_stopping_rounds": 100,
}
if USE_GPU: xgb_base["device"] = "cuda"

print(f"Training XGBoost across {N_SEEDS} seeds ...")
xgb_preds = np.zeros((len(X_test), N_SEEDS))
for s in range(N_SEEDS):
    p = {**xgb_base, "random_state": s * 42}
    m = XGBClassifier(**p)
    m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    xgb_preds[:, s] = m.predict_proba(X_test)[:, 1]
    if s == 0:
        xgb_model_seed0 = m
    else:
        del m
    gc.collect()
xgb_avg = xgb_preds.mean(axis=1)
xgb_auroc = roc_auc_score(y_test, xgb_avg)
print(f"  XGBoost {N_SEEDS}-seed avg AUROC = {xgb_auroc:.4f}    <-- deployment candidate")


In [ ]:
# --- CatBoost ---------------------------------------------------------------
cb_base = {
    **best_params["cb"],
    "loss_function": "Logloss", "eval_metric": "AUC", "verbose": 0,
    "early_stopping_rounds": 100,
}
if USE_GPU: cb_base["task_type"] = "GPU"; cb_base["devices"] = "0"

print(f"Training CatBoost across {N_SEEDS} seeds ...")
cb_preds = np.zeros((len(X_test), N_SEEDS))
for s in range(N_SEEDS):
    p = {**cb_base, "random_seed": s * 42}
    m = CatBoostClassifier(**p)
    m.fit(X_tr, y_tr, eval_set=(X_va, y_va), verbose=0)
    cb_preds[:, s] = m.predict_proba(X_test)[:, 1]
    del m
    gc.collect()
cb_avg = cb_preds.mean(axis=1)
cb_auroc = roc_auc_score(y_test, cb_avg)
print(f"  CatBoost {N_SEEDS}-seed avg AUROC = {cb_auroc:.4f}")


In [ ]:
# --- HistGradientBoosting ---------------------------------------------------
# HistGBM has far fewer knobs and its sklearn defaults are already competitive
# on V7, so we keep it un-tuned. It does its own internal validation via
# validation_fraction=0.1, so we pass the full X_train to it.
print(f"Training HistGBM across {N_SEEDS} seeds ...")
hist_preds = np.zeros((len(X_test), N_SEEDS))
for s in range(N_SEEDS):
    m = HistGradientBoostingClassifier(
        max_iter=1500, learning_rate=0.03, max_leaf_nodes=63,
        min_samples_leaf=200, l2_regularization=1.0,
        random_state=s * 42, early_stopping=True,
        validation_fraction=0.1, n_iter_no_change=100, verbose=0,
    )
    m.fit(X_train, y_train)
    hist_preds[:, s] = m.predict_proba(X_test)[:, 1]
    del m
    gc.collect()
hist_avg = hist_preds.mean(axis=1)
hist_auroc = roc_auc_score(y_test, hist_avg)
print(f"  HistGBM {N_SEEDS}-seed avg AUROC = {hist_auroc:.4f}")


### 10.4 Scipy-optimised blending (Nelder-Mead)

Optimises the four blend weights subject to weights in [0.05, 0.50] and summing to 1.


In [ ]:
# ── 10.4 Scipy-optimized blending + reference-weight blend ──────────────────
# We compute the blend two ways:
#   (a) Scipy-optimized weights (Nelder-Mead on this run's component preds)
#   (b) Pinned REFERENCE weights from the V7 final run, persisted in
#       Final Model Results/model_v7_final_metrics.json
#       weights = (LGB 0.192, XGB 0.298, CB 0.299, HistGBM 0.210)
#       test AUROC = 0.794769
# When the re-run produces near-identical component AUROCs, (a) and (b)
# match to ~0.0005. The pinned weights are the authoritative V7 values.

# --- (a) scipy-optimized on this run ---------------------------------------
def neg_auroc(w):
    w = np.clip(w, 0, None)
    w = w / w.sum()
    blend = w[0]*lgb_avg + w[1]*xgb_avg + w[2]*cb_avg + w[3]*hist_avg
    return -roc_auc_score(y_test, blend)

result = minimize(
    neg_auroc, x0=[0.25, 0.25, 0.25, 0.25],
    method="Nelder-Mead", bounds=[(0.05, 0.5)] * 4,
)
w_opt = result.x / result.x.sum()
blend_preds = (w_opt[0]*lgb_avg + w_opt[1]*xgb_avg +
               w_opt[2]*cb_avg  + w_opt[3]*hist_avg)
blend_auroc = roc_auc_score(y_test, blend_preds)

# --- (b) reference blend — pinned weights from the V7 final run ────────────
REF_WEIGHTS = np.array([0.192, 0.298, 0.299, 0.210])   # LGB, XGB, CB, HistGBM
REF_WEIGHTS = REF_WEIGHTS / REF_WEIGHTS.sum()          # (already sums to 0.999)
ref_blend_preds = (REF_WEIGHTS[0]*lgb_avg + REF_WEIGHTS[1]*xgb_avg +
                   REF_WEIGHTS[2]*cb_avg  + REF_WEIGHTS[3]*hist_avg)
ref_blend_auroc = roc_auc_score(y_test, ref_blend_preds)

print("=" * 64)
print("ENSEMBLE BLEND — V7 (scipy-optimized vs. pinned reference)")
print("=" * 64)
print("                        Scipy-opt      Reference (pinned)")
print(f"  LightGBM weight :      {w_opt[0]:.3f}           {REF_WEIGHTS[0]:.3f}")
print(f"  XGBoost  weight :      {w_opt[1]:.3f}           {REF_WEIGHTS[1]:.3f}")
print(f"  CatBoost weight :      {w_opt[2]:.3f}           {REF_WEIGHTS[2]:.3f}")
print(f"  HistGBM  weight :      {w_opt[3]:.3f}           {REF_WEIGHTS[3]:.3f}")
print(f"\n  Blended AUROC   :      {blend_auroc:.6f}      {ref_blend_auroc:.6f}")
print(f"  XGBoost solo    :      {xgb_auroc:.6f}      (deployment candidate)")
print(f"  Reference blend :      0.794769        (0.7931 solo XGB)")


### 10.4b Reference validator — against `model_v7_final_metrics.json`

Compares this run's per-model AUROCs against the authoritative values
archived in `Final Model Results/model_v7_final_metrics.json`. Deltas of
±0.002 are within seed-level variance at `N_SEEDS = 3`. Larger deltas
typically mean Optuna found a different local optimum (re-run Section 10.2
with `FORCE_RETUNE = True` and more trials) or that library versions have
drifted.


In [ ]:
# ── 10.4b Reference validator ───────────────────────────────────────────────
import json
from pathlib import Path

REF_PATH = BASE_DIR / "Final Model Results" / "model_v7_final_metrics.json"
if REF_PATH.exists():
    ref = json.loads(REF_PATH.read_text())
    targets = {
        "LightGBM": ref["models"]["LightGBM"]["AUROC"],
        "XGBoost":  ref["models"]["XGBoost"]["AUROC"],
        "CatBoost": ref["models"]["CatBoost"]["AUROC"],
        "HistGBM":  ref["models"]["HistGBM"]["AUROC"],
        "Blend":    ref["test_auroc"],
    }
    achieved = {
        "LightGBM": lgb_auroc,
        "XGBoost":  xgb_auroc,
        "CatBoost": cb_auroc,
        "HistGBM":  hist_auroc,
        "Blend":    ref_blend_auroc,   # use reference-weight blend for fair comparison
    }
    print("=" * 64)
    print("REFERENCE VALIDATOR — this run vs. V7 pinned metrics")
    print("=" * 64)
    print(f"  {'Model':<10} {'Target':>8} {'Achieved':>10} {'Δ':>8}  Verdict")
    print(f"  {'-'*10} {'-'*8} {'-'*10} {'-'*8}  {'-'*7}")
    for k in ["LightGBM", "XGBoost", "CatBoost", "HistGBM", "Blend"]:
        t, a = targets[k], achieved[k]
        delta = a - t
        ok = "✓" if abs(delta) < 0.003 else ("~" if abs(delta) < 0.008 else "✗")
        print(f"  {k:<10} {t:>8.4f} {a:>10.4f} {delta:>+8.4f}  {ok}")
    print(f"\n  Reference stability: {ref['stability_mean_auroc']:.4f} ± "
          f"{ref['stability_std_auroc']:.4f}  (5-fold CV)")
    print(f"  Reference n_train / n_test: {ref['n_train']:,} / {ref['n_test']:,}")
    print(f"  This run   n_train / n_test: {len(X_train):,} / {len(X_test):,}")
else:
    print(f"[info] reference metrics not found at {REF_PATH} — skipping validator")


### 10.3 Blend-weight donut (for the presentation)

The blend weights gravitate toward XGBoost and CatBoost (≈ 30% each) and away from LightGBM and HistGBM (≈ 20% each) — matching our expectation that XGBoost and CatBoost produce the most orthogonal prediction patterns.


In [ ]:
# ── 10.3 Ensemble blend weights — donut chart ──────────────────────────────
fig, ax = plt.subplots(figsize=(7, 7))
wedges, _, autotexts = ax.pie(
    w_opt, labels=["LightGBM", "XGBoost", "CatBoost", "HistGBM"],
    colors=[PALETTE["teal"], PALETTE["mint"], PALETTE["gold"], PALETTE["coral"]],
    autopct=lambda p: f"{p:.0f}%",
    startangle=90, wedgeprops=dict(width=0.4, edgecolor="white", linewidth=3),
    textprops={"fontsize": 13, "fontweight": "bold"},
)
for t in autotexts:
    t.set_color("white")
ax.set_title(f"Scipy-Optimized Blend — AUROC = {blend_auroc:.4f}",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("ensemble_blend_weights.png", dpi=150, bbox_inches="tight")
plt.show()


---
## 11. Results

### 11.1 Final XGBoost performance — ROC + calibration

The selected XGBoost model delivers both ranking power and trustworthy probabilities — essential for downstream clinical decision-support — while imposing only a single-model maintenance footprint.


In [ ]:
# ── 11.1 Final XGBoost — ROC, PR, calibration, confusion matrix ────────────
best_preds = xgb_avg      # deployment candidate
best_auroc = xgb_auroc
best_name  = "XGBoost (V7)"

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# A) ROC
fpr, tpr, _ = roc_curve(y_test, best_preds)
axes[0, 0].plot(fpr, tpr, color=PALETTE["teal"], lw=2.5,
                label=f"{best_name}   AUROC = {best_auroc:.4f}")
axes[0, 0].plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5)
axes[0, 0].set_xlabel("False Positive Rate")
axes[0, 0].set_ylabel("True Positive Rate")
axes[0, 0].set_title("ROC Curve — Test Set", fontsize=14, fontweight="bold")
axes[0, 0].legend(fontsize=11)

# B) PR
prec, rec, _ = precision_recall_curve(y_test, best_preds)
ap = average_precision_score(y_test, best_preds)
axes[0, 1].plot(rec, prec, color=PALETTE["coral"], lw=2.5,
                label=f"{best_name}   AP = {ap:.4f}")
axes[0, 1].axhline(y_test.mean(), color="gray", ls="--",
                   label=f"Baseline = {y_test.mean():.3f}")
axes[0, 1].set_xlabel("Recall")
axes[0, 1].set_ylabel("Precision")
axes[0, 1].set_title("Precision-Recall Curve — Test Set",
                     fontsize=14, fontweight="bold")
axes[0, 1].legend(fontsize=11)

# C) Calibration
prob_true, prob_pred = calibration_curve(y_test, best_preds, n_bins=10, strategy="quantile")
brier = brier_score_loss(y_test, best_preds)
axes[1, 0].plot(prob_pred, prob_true, "o-",
                color=PALETTE["teal"], lw=2.5, markersize=8,
                label=f"{best_name}   Brier = {brier:.4f}")
axes[1, 0].plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5, label="Perfect")
axes[1, 0].set_xlabel("Mean predicted probability")
axes[1, 0].set_ylabel("Fraction of positives")
axes[1, 0].set_title("Calibration (Reliability Diagram)",
                     fontsize=14, fontweight="bold")
axes[1, 0].legend(fontsize=11)

# D) Confusion matrix at Youden-J threshold
j_fpr, j_tpr, j_thr = roc_curve(y_test, best_preds)
j = j_tpr - j_fpr
opt_thr = j_thr[np.argmax(j)]
y_pred = (best_preds >= opt_thr).astype(int)
cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[1, 1],
            xticklabels=["No Readmit", "Readmit"],
            yticklabels=["No Readmit", "Readmit"])
axes[1, 1].set_xlabel("Predicted")
axes[1, 1].set_ylabel("Actual")
axes[1, 1].set_title(f"Confusion Matrix (threshold = {opt_thr:.3f})",
                     fontsize=14, fontweight="bold")

plt.tight_layout()
plt.savefig("fig_z_roc_cal.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nOptimal Youden-J threshold: {opt_thr:.4f}")
print(classification_report(y_test, y_pred,
                            target_names=["No Readmit", "Readmit"]))


### 11.2 Final model comparison — V1 vs V6 vs V7 vs Feature Expansion Version

**Table 2.** Final model comparison across dataset versions — best AUROC, model type, interpretability, and deployment complexity.

| Metric | V1 (21 feat) | V6 (40 feat) | V7 (50 feat) | Feature Expansion Version (368 feat) |
|---|---|---|---|---|
| **Best AUROC** | 0.709 | 0.778 | **0.795** | 0.800 |
| **Best model** | Logistic Reg. | Stacking | **4-GBM Blend** | 4-GBM Blend |
| **Interpretability** | High | Moderate | **High** | Low |
| **Deploy complexity** | Low | High | **Low** | Very high |

> **V7 captures 99.4% of the Feature Expansion Version's performance with a compact, interpretable feature set.** Additional feature engineering beyond the core clinical domains yielded only marginal improvement (+0.005 AUROC).


In [ ]:
# ── 11.2 Final summary table (presentation-ready) ──────────────────────────
summary = pd.DataFrame({
    "Model Version":   ["V1 (Baseline)", "V6 (40 feat)", "V7 (50 feat, deployed)",
                        "Feature Expansion Version (368 feat)"],
    "N Features":      [21, 40, 50, 368],
    "Best AUROC":      [0.709, 0.778, 0.795, 0.800],
    "Best Model":      ["Logistic Reg.", "Stacking", "4-GBM Blend", "4-GBM Blend"],
    "Interpretability": ["High", "Moderate", "High", "Low"],
    "Deploy Complexity": ["Low", "High", "Low", "Very High"],
})
print("=" * 92)
print("FINAL MODEL COMPARISON")
print("=" * 92)
print(summary.to_string(index=False))
print("\nV7 selected as deployment frontier: 99.4% of the Feature Expansion "
      "Version's performance at 13.6% of the feature count.")


### 11.3 Benchmark comparison

**Table 3.** Benchmark comparison against published 30-day all-cause readmission models on MIMIC-family data.

| Study | Method | AUROC |
|---|---|---|
| van Walraven et al. (2010) | LACE clinical index | 0.684 |
| Huang et al. (2020) | ClinicalBERT + clinical notes | 0.714 |
| Literature baselines | Single LightGBM / XGBoost | ≈ 0.76 |
| **This work (V7, selected)** | **XGBoost, 50 features** | **0.793** |
| This work (V7, ensemble) | 4-GBM blend (not deployed) | 0.795 |

**Key takeaways**

- **Outperforms traditional scores:** the V7 XGBoost improves over LACE by +0.111 AUROC (a 16% relative gain).
- **Exceeds NLP-based approaches:** higher AUROC than ClinicalBERT *without* clinical notes or NLP pipelines.
- **Structured data only:** 50 interpretable engineered features from structured EHR data — no notes, imaging, or temporal graphs.
- **Clinically deployable:** the parsimonious feature set maps directly to actionable care interventions at discharge.

*Sources: van Walraven et al. (CMAJ, 2010); Huang et al. (arXiv, 2020).*


In [ ]:
# ── 11.3 Benchmark comparison figure ───────────────────────────────────────
benchmarks = {
    "LACE (2010)":           0.684,
    "ClinicalBERT (2020)":   0.714,
    "Lit. LightGBM/XGB":     0.760,
    "Our XGBoost (V7)":      0.793,
    "Our 4-GBM blend (V7)":  0.795,
}

fig, ax = plt.subplots(figsize=(10, 4.5))
colors = [PALETTE["gray"], PALETTE["gray"], PALETTE["gray"],
          PALETTE["teal"], PALETTE["mint"]]
bars = ax.barh(list(benchmarks.keys()), list(benchmarks.values()),
               color=colors, edgecolor="white", linewidth=1.5)
for i, v in enumerate(benchmarks.values()):
    ax.text(v + 0.003, i, f"{v:.3f}", va="center",
            fontsize=11, fontweight="bold")
ax.set_xlim(0.60, 0.82)
ax.axvline(0.684, color=PALETTE["gray"], ls=":", lw=1, alpha=0.5)
ax.set_xlabel("Test AUROC on MIMIC-family data", fontsize=12)
ax.set_title("Benchmark Comparison: 30-Day Readmission Prediction",
             fontsize=14, fontweight="bold")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("benchmark_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


---
## 12. SHAP Interpretability — Global + Patient-Level Explanations

### 12.1 Top 7 features by mean |SHAP|

| Rank | Feature | Clinical interpretation |
|---|---|---|
| 1 | **LOS Trend 180 d** | Rising stay lengths over 6 months — strongest signal of decompensation. |
| 2 | **DRG code** | Clinical mix carries enormous predictive power (11–66% readmission range). |
| 3 | **Late Order Rate** | Operational chaos during the stay predicts post-discharge failure. |
| 4 | **Primary Dx Chapter** | Disease category shapes the baseline risk profile. |
| 5 | **Prior Admits 6m²** | Acceleration in utilisation compounds risk non-linearly. |
| 6 | **Last DRG + Disposition** | Prior discharge pathway predicts future readmission patterns. |
| 7 | **Discharge Location** | Where the patient goes post-discharge directly impacts readmission risk. |

Readmission risk is inherently multifactorial — **no single feature dominates**. The top 7 features map directly to interventions a discharge team can act upon.


In [ ]:
# ── 12.1 SHAP values on XGBoost (tree explainer, matches report Fig. 10) ──
import shap

# Sample for speed
shap_sample = X_test.sample(n=min(5000, len(X_test)), random_state=42)
explainer = shap.TreeExplainer(xgb_model_seed0)
shap_values = explainer.shap_values(shap_sample)
if isinstance(shap_values, list):       # binary classification -> list of 2
    shap_vals = shap_values[1]
else:
    shap_vals = shap_values

print(f"SHAP values computed for {len(shap_sample):,} test samples "
      f"across {shap_vals.shape[1]} features.")


In [ ]:
# ── 12.2 Top-7 features — presentation-ready bar chart ─────────────────────
mean_abs = np.abs(shap_vals).mean(axis=0)
feat_imp = (pd.DataFrame({"feature": shap_sample.columns, "mean_abs_shap": mean_abs})
              .sort_values("mean_abs_shap", ascending=False))

top7 = feat_imp.head(7)
fig, ax = plt.subplots(figsize=(10, 5))
colors = TOP5_COLORS + [PALETTE["teal"], PALETTE["gold"]]
ax.barh(range(6, -1, -1), top7["mean_abs_shap"].values, color=colors)
ax.set_yticks(range(6, -1, -1))
ax.set_yticklabels(top7["feature"].values, fontsize=11)
ax.set_xlabel("Mean |SHAP value|", fontsize=12)
ax.set_title("Top 7 Features — SHAP Global Importance (V7)",
             fontsize=14, fontweight="bold")
for i, v in enumerate(top7["mean_abs_shap"].values):
    ax.text(v + 0.0003, 6 - i, f"{v:.4f}", va="center",
            fontsize=10, fontweight="bold")
plt.tight_layout()
plt.savefig("fig_g_shap7.png", dpi=150, bbox_inches="tight")
plt.show()

print(top7.to_string(index=False))


In [ ]:
# ── 12.3 SHAP beeswarm — top 7 features ────────────────────────────────────
top7_names = top7["feature"].tolist()
top7_idx   = [list(shap_sample.columns).index(f) for f in top7_names]
shap_top7  = shap_vals[:, top7_idx]
data_top7  = shap_sample[top7_names]

fig, ax = plt.subplots(figsize=(12, 5))
shap.summary_plot(shap_top7, data_top7, feature_names=top7_names,
                  max_display=7, show=False, plot_size=None)
plt.title("Top-7 SHAP Beeswarm — V7 XGBoost",
          fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("shap_top7_beeswarm.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── 12.4 Patient-level SHAP waterfall ──────────────────────────────────────
# Find the first actually-readmitted patient in the SHAP sample
idx_to_pos = {idx: pos for pos, idx in enumerate(X_test.index)}
sample_positions = [idx_to_pos[i] for i in shap_sample.index]

sample_idx = next(
    (i for i in range(len(shap_sample)) if y_test[sample_positions[i]] == 1), 0
)
actual = "Readmitted" if y_test[sample_positions[sample_idx]] == 1 else "Not Readmitted"

patient_shap = pd.Series(shap_vals[sample_idx], index=shap_sample.columns)
top_contribs = patient_shap.abs().sort_values(ascending=False).head(7)

print(f"Patient-level SHAP explanation (sample index {sample_idx}):")
print(f"  Actual outcome: {actual}")
print("\n  Top-7 SHAP contributions:")
for feat in top_contribs.index:
    val = patient_shap[feat]
    direction = "RISK UP  " if val > 0 else "risk down"
    print(f"    {feat:30s}  SHAP={val:+.4f}  ({direction})")

base_val = (explainer.expected_value[1]
            if isinstance(explainer.expected_value, (list, np.ndarray))
            else explainer.expected_value)
fig = plt.figure(figsize=(12, 5))
shap.plots.waterfall(shap.Explanation(
    values=shap_vals[sample_idx][top7_idx],
    base_values=base_val,
    data=shap_sample.iloc[sample_idx][top7_names].values,
    feature_names=top7_names,
), max_display=7, show=False)
plt.title("Patient-Level SHAP Waterfall — Top-7 Features",
          fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("shap_patient_waterfall.png", dpi=150, bbox_inches="tight")
plt.show()


---
## 13. Answering the Research Questions

**RQ1 — Which features are most predictive of 30-day readmissions?**
LOS trend (180 d), DRG code, discharge location, prior-admission history, and clinical interactions are the top predictors. A **core set of 50 features captures 99.4% of predictive performance**.

**RQ2 — Which modeling approach achieves the best predictive performance?**
**XGBoost (AUROC 0.793)** was selected over the top-performing 4-GBM blended ensemble (AUROC 0.795) because the negligible 0.002 gap does not justify the added complexity, and XGBoost preserves full feature interpretability for actionable insights.

**RQ3 — Can interpretable ML methods provide actionable insights?**
Yes. **SHAP delivers patient-level explanations** — providers see which factors drive each individual's risk. Key operational levers: LOS monitoring, medication reconciliation, and early follow-up for frequent admitters.


---
## 14. Discussion

### 14.1 Clinical implications — four actionable features

Rather than interpret every SHAP variable, we focus on the four features whose SHAP magnitude and clinical actionability are jointly greatest.

**180-day LOS trend (0.042).** Best read as a compressed biomarker of *disease trajectory* rather than an independent cause. Patients whose recent admissions have become progressively longer are decompensating: stabilisation takes longer, functional reserve is lower at separation, the interval between hospitalisations is shortening. A rising 180-day trend should prompt a pre-discharge geriatric or palliative-care consult for patients above the cohort 90th percentile, and proactive escalation to complex-care management before a fourth admission accrues. *Caveat:* the feature runs high for socially isolated patients whose discharge is delayed for logistical reasons — treating it as a purely clinical signal risks conflating social need with medical severity.

**DRG code (0.030).** Encodes both the admission reason and the CMS complexity weighting. High-risk DRGs (heart failure, COPD exacerbation, septicaemia, AKI) warrant disease-specific bundles (the HFSA heart-failure bundle — weight monitoring, diuretic titration, 7-day follow-up — is representative) and automated pharmacist-led medication reconciliation. *Caveat:* DRG is a billing construct as much as a clinical one; coding conventions and up-coding inflate or deflate the signal in ways the model cannot distinguish from true severity.

**Squared prior-admission count (6 m) (0.011).** Captures a non-linear dose-response: the risk increment between 0 → 1 prior admission is moderate, while 3 → 4 is large. Identifies patients for whom standard discharge planning has already failed; they should enter intensive case-management or community-paramedicine, with an ambulatory complex-care appointment and integrated social-work review within 7 days. *Caveat — the biggest equity concern in the ranking:* high prior utilisation disproportionately reflects poor outpatient access, unstable housing, and insurance gaps, so a score that weights this feature heavily will assign higher risk to exactly the subgroups whose elevated risk is socially rather than biologically driven. **Pairing the score to additional resources, rather than to reduced downstream care, is the only deployment posture that converts the prediction into an equitable intervention.**

**Current discharge location (0.009).** The most directly actionable feature because it sits at the moment the care team still controls. Hospice discharge readmits at 3.9% (terminal-care framework actively avoids admission); psychiatric-facility discharge readmits at 50.3% (severe medical comorbidity + transfer rules that route decompensation back to the index hospital). High-risk destinations should trigger a warm hand-off + destination-specific medication-reconciliation templates. *Caveat:* discharge location is partly endogenous to bed availability and payer rules.

**Read together,** these four features describe a prediction problem whose signal lives at the interface between the patient and the care system. The model is not predicting who will become sicker in isolation; it is predicting whose combination of clinical trajectory and care hand-off is fragile. **Readmission reduction cannot be achieved by risk-scoring alone** — the highest-leverage points are system points (disposition, post-acute hand-off, complex-care enrolment), and the clinical utility of any score is bounded by whether those points can be reached and modified operationally.

### 14.2 Tabular boosting vs. deep learning

Gradient boosting dominates deep tabular architectures on this problem — reinforcing recent benchmarking evidence. Practical consequence: hospitals can obtain state-of-the-art performance without GPU infrastructure, specialised MLOps tooling, or deep-learning-specific governance — all of which add cost and friction to clinical deployment.

### 14.3 Limitations

1. **Single hospital system.** Data from Beth Israel Deaconess (Boston) only — may not generalise to community hospitals, rural settings, or non-US health systems without recalibration.
2. **No unstructured clinical notes.** Physician notes, discharge summaries, and radiology reports were not used; NLP-based features could further improve predictions.
3. **No social determinants of health.** Housing stability, social support, health literacy, and transportation are known risk factors but unavailable in MIMIC-IV.
4. **Diminishing returns.** Expanding features beyond the core clinical set yielded only marginal improvement — suggesting a ceiling with the available structured data.
5. **Temporal limitation.** Readmissions at *other* hospitals are not captured in MIMIC-IV, biasing the label toward within-system utilisation.


---
## 15. Future Work

1. **External validation.** Test on data from other hospital systems to assess generalisability.
2. **NLP integration.** Incorporate unstructured clinical notes using text mining for richer patient context.
3. **Real-time deployment.** Develop an API / dashboard for real-time risk scoring within EHR workflows at discharge.
4. **Fairness analysis.** Evaluate performance across demographic subgroups to identify and mitigate biases.
5. **Causal inference.** Move beyond prediction to identify causal factors enabling more effective interventions.
6. **Cost-effectiveness.** Quantify the economic impact of implementing the prediction model in hospital operations.


---
## 16. Contributions & Conclusions

### 16.1 Four contributions

1. **Reproducible MIMIC-IV Medicare cohort + staged V1 → V7 feature-engineering recipe** that other researchers can extend.
2. **Empirical evidence** that a **single-model XGBoost trained on 50 features** matches or surpasses traditional clinical scores and notes-based deep-learning pipelines while remaining fully interpretable. The additional 0.002 AUROC from a 4-GBM blend does **not** justify its operational overhead.
3. **SHAP-based explanation layer** that converts individual predictions into ranked contributing factors — a bridge between model output and clinician decision.
4. **Quantification of the diminishing-returns frontier** at ≈ 50 curated variables: V7 captures ≈ 99.4% of the expanded 368-feature model's signal with 1/7 the features.

### 16.2 Conclusion

A **single XGBoost model trained on 50 curated features from MIMIC-IV v3.1 predicts 30-day all-cause readmission in Medicare patients with a test AUROC of 0.793** — a +0.109 improvement over LACE and +0.079 over ClinicalBERT. The model is interpretable at both global and individual levels, well calibrated in the operating range that matters for care-coordination triage, and simple enough to be embedded in existing EHR workflows without additional infrastructure.

**Next steps:** external validation on multi-hospital data, fairness analysis across demographic subgroups, and prospective evaluation of the downstream intervention pathway.

---

### Acknowledgments

This work was completed as the capstone project for the MS in Data Science & Artificial Intelligence at Florida International University, under the mentorship of **Dr. Christian Poellabauer**. We gratefully acknowledge the MIT Laboratory for Computational Physiology and the PhysioNet team for maintaining and curating the MIMIC-IV database.


---
## 17. References

1. Johnson, A.E.W., et al. (2023). *MIMIC-IV, a freely accessible electronic health record dataset.* Scientific Data 10, 1.
2. van Walraven, C., et al. (2010). *Derivation and validation of an index to predict early death or unplanned readmission after discharge (LACE).* CMAJ, 182(6), 551–557.
3. Huang, K., Altosaar, J., & Ranganath, R. (2020). *ClinicalBERT: Modeling clinical notes and predicting hospital readmission.* arXiv:1904.05342.
4. Gorishniy, Y., Rubachev, I., Khrulkov, V., & Babenko, A. (2021). *Revisiting deep learning models for tabular data.* NeurIPS.
5. Chen, T., & Guestrin, C. (2016). *XGBoost: A scalable tree boosting system.* KDD '16, 785–794.
6. Ke, G., et al. (2017). *LightGBM: A highly efficient gradient boosting decision tree.* NeurIPS.
7. Prokhorenkova, L., et al. (2018). *CatBoost: Unbiased boosting with categorical features.* NeurIPS.
8. Lundberg, S., & Lee, S.-I. (2017). *A unified approach to interpreting model predictions (SHAP).* NeurIPS.
9. Centers for Medicare & Medicaid Services. (2024). *Hospital Readmissions Reduction Program (HRRP).*
10. Charlson, M.E., et al. (1987). *A new method of classifying prognostic comorbidity in longitudinal studies.* Journal of Chronic Diseases, 40(5), 373–383.

---

*Thiago Bandeira & Armando Gonzalez — Florida International University — April 2026*
